In [1]:
import numpy as np
import pandas as pd
import io

DATASET PARSING

In [2]:
def parse_conllu_dataset(file_path):
    corpus = []
    current_sentence = []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()

            if not line:
                if current_sentence:
                    corpus.append(current_sentence)
                    current_sentence = []
            elif line.startswith('#'):
                continue
            else:
                parts = line.split('\t')
                if len(parts) >= 4:
                    word = parts[1]
                    tag = parts[3]
                    current_sentence.append((word, tag))

    if current_sentence:
        corpus.append(current_sentence)

    # Rebuild a flat DataFrame for our Pandas training function
    flat_data = [(w, t) for sentence in corpus for w, t in sentence]
    raw_df = pd.DataFrame(flat_data, columns=['Word', 'Tag'])

    return corpus, raw_df

TRAINING --> LAPLACE SMOOTHING

In [3]:
def train_hmm_laplace(corpus, raw_df, alpha=0.1):
    words = raw_df['Word'].tolist()
    tags = raw_df['Tag'].tolist()

    vocab = list(set(words))
    states = list(set(tags))

    transitions = []
    start_tags = []
    for sentence in corpus:
        start_tags.append(sentence[0][1])
        for i in range(len(sentence) - 1):
            transitions.append((sentence[i][1], sentence[i+1][1]))

    pi_counts = pd.Series(start_tags).value_counts()
    # I added alpha to all states to ensure no zero probabilities
    pi_smoothed = (pi_counts.reindex(states, fill_value=0) + alpha)
    pi = pi_smoothed / pi_smoothed.sum()

    # smoothen the transition matrix, aplha and emmission matrix
    trans_df = pd.DataFrame(transitions, columns=['From', 'To'])
    transition_counts = pd.crosstab(trans_df['From'], trans_df['To'])
    transition_counts = transition_counts.reindex(index=states, columns=states, fill_value=0)

    A_smoothed = transition_counts + alpha
    A = A_smoothed.div(A_smoothed.sum(axis=1), axis=0)

    emission_counts = pd.crosstab(raw_df['Tag'], raw_df['Word'])
    emission_counts = emission_counts.reindex(index=states, fill_value=0)

    B_smoothed = emission_counts + alpha
    B = B_smoothed.div(B_smoothed.sum(axis=1), axis=0)

    return pi, A, B, states, vocab

MORPHOLOGICAL FALLBACK

In [4]:
def get_oov_emission(word, states):
    """
    If a word is Out-Of-Vocabulary, guess its POS tag based on common English suffixes.
    Returns a probability distribution array aligned with 'states'.
    """
    N = len(states)
    probs = np.ones(N) # begin with uniform baseline

    word_lower = word.lower()

    # Educated Guess define
    guesses = {}
    if word_lower.endswith('ly'):
        guesses['ADV'] = 5.0
    elif word_lower.endswith('ed') or word_lower.endswith('ing'):
        guesses['VERB'] = 5.0
    elif word_lower.endswith('ness') or word_lower.endswith('tion') or word_lower.endswith('ment'):
        guesses['NOUN'] = 5.0
    elif word_lower.endswith('ous') or word_lower.endswith('ful') or word_lower.endswith('able'):
        guesses['ADJ'] = 5.0
    elif word_lower.endswith('s'):
        guesses['NOUN'] = 3.0
        guesses['VERB'] = 3.0

    # apply the boosts to our probability array
    for i, state in enumerate(states):
        if state in guesses:
            probs[i] += guesses[state]

    # normalize so they sum to 1
    return probs / np.sum(probs)

DECODING

In [5]:
def viterbi_robust(obs_seq, pi, A, B, states, vocab):
    N = len(states)
    T = len(obs_seq)

    pi_np = np.log(pi.values)
    A_np = np.log(A.values)

    viterbi_table = np.zeros((N, T))
    backpointer = np.zeros((N, T), dtype=int)

    # init
    first_word = obs_seq[0]
    if first_word in vocab:
        B_init = np.log(B[first_word].values)
    else:
        B_init = np.log(get_oov_emission(first_word, states))

    viterbi_table[:, 0] = pi_np + B_init

    # recurse
    for t in range(1, T):
        word = obs_seq[t]
        if word in vocab:
            B_curr = np.log(B[word].values)
        else:
            B_curr = np.log(get_oov_emission(word, states))

        for s in range(N):
            trans_probs = viterbi_table[:, t-1] + A_np[:, s]
            best_prev_state = np.argmax(trans_probs)
            viterbi_table[s, t] = trans_probs[best_prev_state] + B_curr[s]
            backpointer[s, t] = best_prev_state

    # backtracking
    best_last_state = np.argmax(viterbi_table[:, T-1])
    best_path = [best_last_state]

    for t in range(T-1, 0, -1):
        best_last_state = backpointer[best_last_state, t]
        best_path.insert(0, best_last_state)

    return [states[i] for i in best_path]

EXECUTION

In [9]:
file_name = 'en_ewt-ud-train.conllu.txt'

print("Parsing dataset...")
corpus, raw_df = parse_conllu_dataset(file_name)

print("Training HMM...")
pi, A, B, states, vocab = train_hmm_laplace(corpus, raw_df, alpha=0.1)

print(f"\nTotal Vocabulary Size: {len(vocab)} words")
print(f"Total POS Tags: {len(states)} tags")

print("\nInitial State Probabilities (Pi)")
print(pi.head(5))

print("\nTransition Matrix (A) [5x5 Slice]")
print(A.iloc[:5, :5])

print("\nEmission Matrix (B) [5x5 Slice]")
print(B.iloc[:5, :5])

Parsing dataset...
Training HMM...

Total Vocabulary Size: 20201 words
Total POS Tags: 18 tags

Initial State Probabilities (Pi)
ADV      0.075173
DET      0.100440
PUNCT    0.035239
PRON     0.252284
CCONJ    0.023123
Name: count, dtype: float64

Transition Matrix (A) [5x5 Slice]
To          ADV       DET     PUNCT      PRON     CCONJ
From                                                   
ADV    0.088778  0.045479  0.172121  0.084142  0.025851
DET    0.015772  0.009883  0.012828  0.005037  0.000620
PUNCT  0.061772  0.064036  0.084963  0.122989  0.109403
PRON   0.053469  0.018970  0.056041  0.027594  0.011309
CCONJ  0.082129  0.097236  0.013028  0.167534  0.000165

Emission Matrix (B) [5x5 Slice]
Word          !        !!       !!!      !!!!     !!!!!
Tag                                                    
ADV    0.000008  0.000008  0.000008  0.000008  0.000008
DET    0.000005  0.000005  0.000005  0.000005  0.000005
PUNCT  0.020655  0.002737  0.002229  0.000824  0.000121
PRON   0.0000

TEST

In [10]:
test_sentences = [
    ["The", "mysterious", "algorithm", "learns", "quickly"],
    ["Time", "flies", "like", "an", "arrow"],
    ["Fruit", "flies", "like", "a", "banana"],
    ["The", "snarfer", "blooped", "loudly", "and", "the", "wombats", "rejoiced"]
]

for sentence in test_sentences:
    predicted_tags = viterbi_robust(sentence, pi, A, B, states, vocab)

    print(f"\nSentence: {sentence}")
    print(f"Predicted POS: {predicted_tags}")


Sentence: ['The', 'mysterious', 'algorithm', 'learns', 'quickly']
Predicted POS: ['DET', 'ADJ', 'NOUN', 'VERB', 'ADV']

Sentence: ['Time', 'flies', 'like', 'an', 'arrow']
Predicted POS: ['PROPN', 'VERB', 'ADP', 'DET', 'NOUN']

Sentence: ['Fruit', 'flies', 'like', 'a', 'banana']
Predicted POS: ['INTJ', 'VERB', 'ADP', 'DET', 'NOUN']

Sentence: ['The', 'snarfer', 'blooped', 'loudly', 'and', 'the', 'wombats', 'rejoiced']
Predicted POS: ['DET', 'NOUN', 'VERB', 'ADV', 'CCONJ', 'DET', 'NOUN', 'VERB']
